In [1]:
import numpy as np
import matplotlib.pyplot as plt

from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.sources import bitSource
from optic.comm.modulation import modulateGray
from optic.comm.metrics import bert
from optic.dsp.core import firFilter, pulseShape, upsample, anorm
from optic.utils import parameters, dBm2W

In [2]:
# =========================
# Simulation parameters
# =========================
SpS = 16
M = 2
Rs = 10e9
Fs = SpS * Rs

Pi_dBm = 3
Pi = dBm2W(Pi_dBm)

# Bit source
paramBits = parameters()
paramBits.nBits = 2**18
paramBits.mode = "random"
paramBits.seed = 123

# Pulse shaping
paramPulse = parameters()
paramPulse.pulseType = "nrz"
paramPulse.SpS = SpS

# MZM
paramMZM = parameters()
paramMZM.Vpi = 2
paramMZM.Vb = -paramMZM.Vpi / 2

# Channel
paramCh = parameters()
paramCh.L = 100
paramCh.alpha = 0.2
paramCh.D = 16
paramCh.Fc = 193.1e12
paramCh.Fs = Fs

# Photodiode
paramPD = parameters()
paramPD.ideal = False
paramPD.B = Rs
paramPD.Fs = Fs
paramPD.seed = 456


def optical_chain(symbTx):
    symbolsUp = upsample(symbTx, SpS)

    pulse = pulseShape(paramPulse)
    sigTx = firFilter(pulse, symbolsUp)
    sigTx = anorm(sigTx)

    # same artificial nonlinearity as the original notebook
    alpha = 0.5
    sigTx = sigTx + alpha * sigTx**3

    Ai = np.sqrt(Pi)
    sigTxo = mzm(Ai, sigTx, paramMZM)
    sigCh = linearFiberChannel(sigTxo, paramCh)
    I_Rx = photodiode(sigCh, paramPD)

    # symbol-rate samples
    I_Rx = I_Rx[0::SpS]
    return I_Rx


# Baseline without DPD
bitsTx = bitSource(paramBits)
symbTx = modulateGray(bitsTx, M, "pam")

I_Rx_no_dpd = optical_chain(symbTx)
BER_no_dpd, Q_no_dpd = bert(I_Rx_no_dpd, bitsTx)

print(f"NO DPD  | Q = {Q_no_dpd:.4f} | BER = {BER_no_dpd:.4e}")

NO DPD  | Q = 2.5850 | BER = 3.3112e-03


In [3]:
import numpy as np

def r1d(x):
    return np.real(np.asarray(x).reshape(-1)).astype(np.float64)


def mp_feat(x, memory=7, orders=(1, 3, 5)):
    x = r1d(x)
    N = len(x)
    start = memory - 1
    cols = []

    for p in orders:
        xp = x ** p
        for m in range(memory):
            cols.append(xp[start - m : N - m])

    return np.column_stack(cols), start


def fit_mp(rx, target, memory=7, orders=(1, 3, 5), ridge=1e-3):
    rx = r1d(rx)
    target = r1d(target)

    Phi, start = mp_feat(rx, memory, orders)
    d = target[start:]

    scale = np.sqrt(np.mean(Phi**2, axis=0) + 1e-12)
    Phi_n = Phi / scale

    w_n = np.linalg.solve(
        Phi_n.T @ Phi_n + ridge * np.eye(Phi_n.shape[1]),
        Phi_n.T @ d
    )

    return {
        "memory": memory,
        "orders": tuple(orders),
        "w": w_n / scale,
        "start": start,
    }


def apply_mp(x, model):
    x = r1d(x)
    Phi, start = mp_feat(x, model["memory"], model["orders"])
    y = x.copy()
    y[start:] = Phi @ model["w"]
    return y


def clip_match(x, ref, clip_sigma=3.0):
    x = r1d(x)
    ref = r1d(ref)

    x = x - np.mean(x)
    ref = ref - np.mean(ref)

    x *= (np.std(ref) + 1e-12) / (np.std(x) + 1e-12)

    mu = np.mean(x)
    sd = np.std(x) + 1e-12
    return np.clip(x, mu - clip_sigma * sd, mu + clip_sigma * sd)






In [4]:
def train_mp_ila(
    train_symb,
    train_bits,
    n_iter=4,
    memory=7,
    orders=(1, 3, 5),
    ridge=1e-3,
    clip_sigma=3.0,
    verbose=True,
):
    x_ref = r1d(train_symb)
    x_pd = x_ref.copy()

    best = None
    hist = []

    for k in range(n_iter):
        y_cur = optical_chain(x_pd)
        ber_cur, q_cur = bert(y_cur, train_bits)

        model = fit_mp(y_cur, x_ref, memory, orders, ridge)

        x_new = apply_mp(x_ref, model)
        x_new = clip_match(x_new, x_ref, clip_sigma)

        y_new = optical_chain(x_new)
        ber_new, q_new = bert(y_new, train_bits)

        hist.append((k + 1, ber_cur, ber_new, q_cur, q_new))

        if verbose:
            print(
                f"iter {k+1:02d} | "
                f"cur BER = {ber_cur:.4e} | new BER = {ber_new:.4e}"
            )

        if best is None or ber_new < best["ber"]:
            best = {
                "iter": k + 1,
                "ber": ber_new,
                "q": q_new,
                "model": model,
            }

        x_pd = x_new

    if verbose:
        print("\nBest training point")
        print(f"iter = {best['iter']}")
        print(f"BER  = {best['ber']:.4e}")
        print(f"Q    = {best['q']:.4f}")

    return best, hist

In [5]:
# -------------------------
# Training data
# -------------------------
paramBits.seed = 3455
bitsTrain = bitSource(paramBits)
symbTrain = modulateGray(bitsTrain, M, "pam")

best_dpd, hist = train_mp_ila(
    train_symb=symbTrain,
    train_bits=bitsTrain,
    n_iter=4,
    memory=7,
    orders=(1, 3, 5),
    ridge=1e-3,
    clip_sigma=3.0,
    verbose=True,
)


# -------------------------
# Test data
# -------------------------
paramBits.seed = 3455
bitsTest = bitSource(paramBits)
symbTest = modulateGray(bitsTest, M, "pam")

# No DPD
I_Rx_no_dpd = optical_chain(symbTest)
BER_no_dpd, Q_no_dpd = bert(I_Rx_no_dpd, bitsTest)

# Volterra DPD
symbTest_dpd = apply_mp(symbTest, best_dpd["model"])
symbTest_dpd = clip_match(symbTest_dpd, symbTest, clip_sigma=3.0)

I_Rx_volterra = optical_chain(symbTest_dpd)
BER_volterra, Q_volterra = bert(I_Rx_volterra, bitsTest)

print("\nTest results")
print(f"NO DPD  | Q = {Q_no_dpd:.4f} | BER = {BER_no_dpd:.4e}")
print(f"Volterra | Q = {Q_volterra:.4f} | BER = {BER_volterra:.4e}")




iter 01 | cur BER = 3.4103e-03 | new BER = 3.9859e-01
iter 02 | cur BER = 3.9859e-01 | new BER = 4.8447e-04
iter 03 | cur BER = 4.8447e-04 | new BER = 2.4033e-03
iter 04 | cur BER = 2.4033e-03 | new BER = 5.6076e-04

Best training point
iter = 2
BER  = 4.8447e-04
Q    = 3.2026

Test results
NO DPD  | Q = 2.5813 | BER = 3.4103e-03
Volterra | Q = 3.2026 | BER = 4.8447e-04
